# Object-Oriented Programming -- Classes and Objects
Topics: what a class/object is, `__init__` and `self`, instance vs class attributes,
methods, encapsulation, inheritance, polymorphism, abstraction, and the `__str__`
dunder method.

## 1. What Is a Class? What Is an Object?
A **class** is a blueprint. It describes what something looks like and what it can do,
but it isn't the thing itself -- a house blueprint doesn't keep anyone warm, and a
cookie cutter isn't a cookie.

An **object** (also called an **instance**) is the actual thing built from that
blueprint. One blueprint, many houses. One cookie cutter, many cookies. Each object
gets its own data, even though they all come from the same class.

Here's the part that makes this less new than it sounds: you've been using objects
since your very first line of Python.

In [1]:
print(type("hello"))    # <class 'str'>   -- "hello" is an object of class str
print(type([1, 2, 3]))   # <class 'list'>  -- this list is an object of class list
print(type(42))          # <class 'int'>

<class 'str'>
<class 'list'>
<class 'int'>


## 2. Defining Your First Class

In [1]:
class Student:
    pass


s1 = Student()
print(s1)            # <__main__.Student object at 0x...>
print(type(s1))      # <class '__main__.Student'>

<class '__main__.Student'>


`class Student:` creates the blueprint. `Student()` builds an object from it.
That object is real -- `type(s1)` proves it's a `Student` -- but it's also empty: it
doesn't have a name, an age, or anything else, because nothing told it what to store
yet. `__init__` is how a class sets up every new object with real data.

## 3. The `__init__` Constructor and `self`
`__init__` is a special method that runs automatically every time you create an
object. `self` is the object being built -- Python passes it in automatically, you
never type it yourself when creating the object.

In [3]:
class Student:
    def __init__(self, name, age, course):
        self.name = name
        self.age = age
        self.course = course



ali = Student("Ali", 20, "Data Engineering")
sara = Student("Sara", 22, "Data Engineering")

print(ali.name, ali.age, ali.course)
print(sara.name, sara.age, sara.course)

Ali 20 Data Engineering
Sara 22 Data Engineering


>
> write `ali = Student("Ali", 20, "Data Engineering")` on the
> board and ask "how many arguments does `__init__` take?" (four: `self, name, age,
> course`) then "how many did we pass?" (three). `self` is filled in *for* you -- it's
> `ali` itself, before `ali` even has a name. That's the one sentence that untangles
> `self` for most students: **`self` just means "the object currently being set up."**
>
> `self.name = name` reads confusingly at first because the same word appears on
> both sides. Read it right-to-left: "take the `name` that was passed in, and store
> it on `self` under the label `name`." The label (left side) doesn't have to match
> the parameter (right side), it's just clearest when it does.

## 4. Objects Are Independent
Each object keeps its own copy of the data. Changing one doesn't touch another.

In [6]:
ali.age = 32

print(ali.age, sara.age)   # 21 22 -- only ali changed

32 22


## 5. Adding Behavior -- Methods
A **method** is a function defined inside a class. It automatically receives `self`,
so it can read or change that specific object's data.

In [8]:
class Student:
    def __init__(self, name, age, course):
        self.name = name
        self.age = age
        self.course = course
        self.grades = []
        

    def introduce(self):
        print(f"Hi, I'm {self.name}, {self.age} years old, studying {self.course}.")

    def add_grade(self, grade):
        self.grades.append(grade)

    def average_grade(self):
        if not self.grades:
            return 0
        return sum(self.grades) / len(self.grades)

ali = Student("Ali Aslam", 20, "Data Engineering")
print(ali.name)
ali.introduce()
ali.add_grade(85)
ali.add_grade(90)
ali.add_grade(78)
print(f"{ali.name}'s average grade: {ali.average_grade():.1f}")

Ali Aslam
Hi, I'm Ali Aslam, 20 years old, studying Data Engineering.
Ali Aslam's average grade: 84.3


## 6. Instance Attributes vs Class Attributes
An **instance attribute** (`self.name`) belongs to one object. A **class attribute**
is defined directly in the class body and is shared by every object made from it.

In [10]:
class Student:
    school_name = "Saylani Mass IT Training"   # class attribute -- one copy, shared

    def __init__(self, name, age, course):
        self.name = name     # instance attribute -- one copy per object
        self.age = age
        self.course = course


ali = Student("Ali", 20, "Data Engineering")
sara = Student("Sara", 22, "Data Engineering")

print(ali.school_name, "|", sara.school_name)

Student.school_name = "Bahria Karachi (Updated Campus)"
print(ali.school_name, "|", sara.school_name)   # both changed -- proof it's shared

Saylani Mass IT Training | Saylani Mass IT Training
Bahria Karachi (Updated Campus) | Bahria Karachi (Updated Campus)


> the classic gotcha:** a *mutable* class attribute (a list or
> dict) is a trap, because every object shares the exact same object, not just the
> same value. This bites almost everyone at least once.

In [11]:
class Bad:
    items = []   # DANGER: one shared list, not a fresh one per student

    def add(self, item):
        self.items.append(item)


a = Bad()
b = Bad()
a.add("apple")
print(b.items)   # ['apple'] -- b got "apple" too, even though we only touched a!

['apple']


Why: `.append()` mutates the *one* list object stored on the class -- it never
creates a new list for `a` or `b`. The fix is to create the list fresh inside
`__init__` (`self.items = []`) so each object gets its own, exactly like `Student`
does above with `self.grades = []`.

## 7. Encapsulation -- Controlling Access to Data
**Encapsulation** means bundling data with the methods that are allowed to change
it, and limiting direct outside access to that data. Python signals intent with
naming, rather than locking anything down completely:

| Prefix | Meaning | Enforced by Python? |
|---|---|---|
| `name` | public -- anyone can read/write it | -- |
| `_name` | protected -- "internal use", please don't touch from outside | No, just a convention |
| `__name` | private -- name-mangled to `_ClassName__name` | Mostly, but not truly locked |

In [20]:
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner          # public
        self._balance = balance     # protected
        self.__pin = "0000"         # private

    def deposit(self, amount):
        if amount > 0:
            self._balance += amount

    def get_balance(self):
        return self._balance        # controlled, read-only access

    def get_pin(self):
        return elf.__pin
        # print(f'the pin is {self.__pin}')

    def change_pin(self, old_pin, new_pin):
        if old_pin == self.__pin:
            self.__pin = new_pin
            return "PIN updated."
        return "Incorrect PIN."


acc_bilal = BankAccount("Bilal", 5000)
acc_bilal.deposit(1500)
# print(acc_bilal.owner)            # public -- fine
# print(acc_bilal.get_balance())    # 6500 -- reached through a method, not raw access
# acc_bilal._balance = 2500
# print(acc_bilal.get_balance())    
# print(acc_bilal.change_pin("0000", "4321"))
print(f'Bilals account pin is {acc_bilal.get_pin}')

Bilals account pin is <bound method BankAccount.get_pin of <__main__.BankAccount object at 0x0000014805E901A0>>


In [9]:
print(acc._balance)   # 6500 -- "protected" still works from outside, it's a promise, not a lock

try:
    print(acc.__pin)        # this name doesn't exist from outside the class...
except AttributeError as e:
    print("Blocked:", e)

print(acc._BankAccount__pin)   # 4321 -- ...because Python renamed it, it didn't hide it

6500
Blocked: 'BankAccount' object has no attribute '__pin'
4321


> **Teaching note:** the punchline is "Python's privacy is a speed bump, not a
> wall." Compare to Java/C++, where `private` is enforced by the compiler. Python's
> philosophy (sometimes summarized as "we're all consenting adults here") trusts you
> not to reach into `__pin` just because you technically can.

## 8. Inheritance -- Reusing and Extending a Class
**Inheritance** lets one class (the child) start from another class (the parent) and
add or override what it needs, instead of retyping everything. `super().__init__()`
calls the parent's constructor to do the shared setup.

In [32]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def introduce(self):
        print(f"Hi, I'm {self.name}, {self.age} years old.")


class Student(Person):
    def __init__(self, name, age, course):
        super().__init__(name, age)   # let Person handle name/age
        self.course = course

    def introduce(self):              # overrides Person's version
        print(f"Hi, I'm {self.name}, studying {self.course}.")

class student_ali(Student):
    def __init(self,name,age):
        super().__init__(name,age)
    def introduce(self):
        print(f'hi i am only ali here {self.name}' )
    
class Teacher(Person):
    def __init__(self, name, age, subject):
        super().__init__(name, age)
        self.subject = subject

    def introduce(self):
        print(f"Hi, I'm {self.name}, I teach {self.subject}.")


        
ali = Student("Ali", 20, "Data Engineering")
sir_ahmed = Teacher("Ahmed", 35, "Python")

ali.introduce()
sir_ahmed.introduce()

Hi, I'm Ali, studying Data Engineering.
Hi, I'm Ahmed, I teach Python.


In [28]:
# Base classes
class A:
    def method_a(self):
        print("Method from class A")

class B:
    def method_b(self):
        print("Method from class B")

# Derived class with multiple inheritance
class C(A, B):
    def method_c(self):
        print("Method from class C")

# Usage
obj = C()
obj.method_a()  # From A
obj.method_b()  # From B
obj.method_c()  # From C


Method from class A
Method from class B
Method from class C


> **Teaching note:** point out that `Student` and `Teacher` never wrote
> `self.name = name` themselves -- `super().__init__(name, age)` did it for them.
> Without it, both classes would duplicate the exact same two lines, and a future
> change to how `Person` stores a name would need to be copy-pasted everywhere.
> That repeated-code smell is precisely what inheritance exists to remove.

## 9. Polymorphism -- Same Method Call, Different Behavior
"Poly" = many, "morph" = forms. Different classes can implement the *same* method
name their own way. Calling code doesn't need to know or care which class it's
actually holding -- it just calls the method and trusts each object to know how to
handle it.

In [34]:
people = [ali, sir_ahmed, Student("Sara", 22, "Data Engineering")]
std = Student("Sara", 22, "Data Engineering")

print(ali.introduce())
print(sir_ahmed.introduce())
print(std.introduce())


# for person in people:
#     person.introduce()   # same call, three different results

Hi, I'm Ali, studying Data Engineering.
None
Hi, I'm Ahmed, I teach Python.
None
Hi, I'm Sara, studying Data Engineering.
None


> **Teaching note:** the loop body never checks "is this a Student or a
> Teacher?" -- and that's the entire point. Add an `Admin` class tomorrow with its
> own `introduce()`, drop it into `people`, and this loop keeps working with zero
> changes. That's what "polymorphism" buys you in real code.

## 10. Abstraction -- Hiding the "How", Exposing the "What"
**Abstraction** means exposing a simple method name while hiding the messy logic
behind it. You press a car's accelerator pedal without knowing anything about fuel
injection -- the pedal is the abstraction. In Python, `abc.ABC` (Abstract Base Class)
lets you go one step further: *force* every subclass to provide its own version of a
method, because there's no sensible shared default.

In [40]:
from abc import ABC, abstractmethod


class Shape(ABC):
    @abstractmethod
    def area(self):
        pass   # no default -- every shape MUST define its own area()


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.1416 * self.radius ** 2
rect = Rectangle(4, 6)
print(f"{type(rect).__name__} area is {rect.area()}")

circ = Circle(5)
print(f"{type(circ).__name__} area is {circ.area()}")


# shapes = [Rectangle(4, 6), Circle(5)]
# for shape in shapes:
#     print(f"{type(shape).__name__} area: {shape.area():.2f}")

Rectangle area is 24
Circle area is 78.53999999999999


In [13]:
try:
    shape = Shape()   # can't build the blueprint itself, only finished shapes
except TypeError as e:
    print("Blocked:", e)

Blocked: Can't instantiate abstract class Shape without an implementation for abstract method 'area'


`Shape` on its own is deliberately incomplete -- it's a contract ("anything
calling itself a `Shape` must have an `.area()`"), not a usable object. `Rectangle`
and `Circle` fulfill that contract; `Shape` itself can't be built.

## 11. Bonus: `__str__` -- Making `print(object)` Readable
Methods with double underscores on both sides (`__init__`, `__str__`) are called
**dunder methods** ("double underscore"). `__str__` controls what `print()` shows for
an object -- without it, Python falls back to an unhelpful default.

In [14]:
class Student:
    def __init__(self, name, course):
        self.name = name
        self.course = course


ali = Student("Ali", "Data Engineering")
print(ali)   # <__main__.Student object at 0x...> -- technically true, not useful

In [41]:
class Student:
    def __init__(self, name, course):
        self.name = name
        self.course = course

    def __str__(self):
        return f"Student name and course is ({self.name}, {self.course})"


ali = Student("Ali", "Data Engineering")
print(ali)   # Student(Ali, Data Engineering) -- print() calls __str__ automatically

Student name and course is (Ali, Data Engineering)


## 12. Putting It Together -- A Mini Classroom System
Everything from this notebook in one program: inheritance (`Person` -> `Student` /
`Teacher`), encapsulation (private, validated grades), polymorphism (`summary()`
means something different per class), and `__str__` for clean output.

In [42]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __str__(self):
        return f"{self.__class__.__name__}: {self.name} ({self.age})"


class Student(Person):
    def __init__(self, name, age, course):
        super().__init__(name, age)
        self.course = course
        self.__grades = []   # private -- only add_grade() is allowed to touch this

    def add_grade(self, grade):
        if 0 <= grade <= 100:
            self.__grades.append(grade)
        else:
            print(f"Rejected invalid grade: {grade}")

    def average_grade(self):
        return sum(self.__grades) / len(self.__grades) if self.__grades else 0

    def summary(self):
        return f"{self} studying {self.course}, average grade {self.average_grade():.1f}"


class Teacher(Person):
    def __init__(self, name, age, subject):
        super().__init__(name, age)
        self.subject = subject

    def summary(self):
        return f"{self} teaching {self.subject}"


roster = [
    Student("Ali", 20, "Data Engineering"),
    Teacher("Ahmed", 35, "Python"),
    Student("Sara", 22, "Data Engineering"),
]

roster[0].add_grade(85)
roster[0].add_grade(92)
roster[0].add_grade(150)   # invalid -- encapsulation's validation rejects it
roster[2].add_grade(70)

for person in roster:
    print(person.summary())   # polymorphism: Student and Teacher each define summary() differently

Rejected invalid grade: 150
Student: Ali (20) studying Data Engineering, average grade 88.5
Teacher: Ahmed (35) teaching Python
Student: Sara (22) studying Data Engineering, average grade 70.0


## Recap -- Vocabulary Cheat Sheet
- **Class** -- the blueprint (`class Student:`)
- **Object / instance** -- a specific thing built from a class (`ali = Student(...)`)
- **`__init__`** -- runs automatically when an object is created, sets up its data
- **`self`** -- "the object currently being worked on"; Python passes it in for you
- **Instance attribute** -- data unique to one object (`self.name`)
- **Class attribute** -- data shared by every object of that class
- **Method** -- a function defined inside a class, called on an object (`ali.introduce()`)
- **Encapsulation** -- bundling data with the methods allowed to touch it (`_protected`, `__private`)
- **Inheritance** -- a class reusing/extending another class (`class Student(Person):`, `super()`)
- **Polymorphism** -- the same method name behaving differently per class
- **Abstraction** -- exposing a simple interface, hiding (or requiring) the implementation (`abc.ABC`)
- **Dunder method** -- double-underscore method Python calls for you (`__init__`, `__str__`)